# 15. Exposure, replication, and block-preserving resampling

![Exposure and replication](../images/15_exposure_and_replication.svg)

**Learning goals:** distinguish exposure from available support, keep participants separate from trained-model blocks, preserve all four factorial cells during resampling, calculate the completion-gap interaction, and run an outcome-blind prospective simulation with eight blocks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 15
rng = np.random.Generator(np.random.PCG64(SEED))
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Fixed exposure and changing support

Every model receives the same sampled-example exposure. Low and high sequence pools still provide different source support, while frozen and resampled policies provide different temporal-anchor support. Exposure counts optimization draws. Support counts the distinct units available to those draws. Neither count by itself establishes independent evidence.

In [ ]:
n_blocks = 8
training_exposure = 8_192_000
low_sequences, high_sequences = 2_500, 250_000
exposure_by_cell = np.full((n_blocks, 2, 2), training_exposure, dtype=np.int64)
support_by_cell = np.empty((n_blocks, 2, 2), dtype=np.int64)
support_by_cell[:, 0, :] = low_sequences
support_by_cell[:, 1, :] = high_sequences
assert np.unique(exposure_by_cell).tolist() == [training_exposure]
assert np.all(support_by_cell[:, 0, :] < support_by_cell[:, 1, :])
print(f"fixed exposure={training_exposure:,}; sequence support={low_sequences:,} or {high_sequences:,}")

## 2. Participants and model blocks are different axes

The outcome tensor has shape `(block, sequence_support, window_policy, participant)`. The last axis contains repeated measurements on the same people. The first axis contains eight matched four-cell training blocks. Primary model-level inference reduces the participant axis and produces one interaction per block.

In [ ]:
B, P = 8, 308
y_expectation = np.array([[0.42, 0.56], [0.58, 0.60]])
c_expectation = np.array([[0.36, 0.44], [0.48, 0.51]])
participant = rng.normal(0, 0.05, size=(1, 1, 1, P))
block = rng.normal(0, 0.012, size=(B, 1, 1, 1))
cell_noise_y = rng.normal(0, 0.010, size=(B, 2, 2, 1))
cell_noise_c = rng.normal(0, 0.010, size=(B, 2, 2, 1))
scores_y = np.clip(y_expectation[None, :, :, None] + participant + block + cell_noise_y
                   + rng.normal(0, 0.025, (B, 2, 2, P)), 0, 1)
scores_c = np.clip(c_expectation[None, :, :, None] + 0.7 * participant + block + cell_noise_c
                   + rng.normal(0, 0.025, (B, 2, 2, P)), 0, 1)
assert scores_y.shape == scores_c.shape == (8, 2, 2, 308)
print("Y and C shapes:", scores_y.shape, scores_c.shape)

## 3. Primary and completion-gap interactions

The function below averages participants within every cell and returns one difference-in-differences value per block. Applying it to $Y$ gives $I_r$. Applying it to $G=Y-C$ gives $J_r$, the interaction not shared with independent-factor completion. $Y$ and $C$ must remain paired by block, cell, and participant.

In [ ]:
def block_interactions(values):
    means = np.asarray(values, dtype=np.float64).mean(axis=-1)
    return (means[:, 1, 1] - means[:, 0, 1]) - (means[:, 1, 0] - means[:, 0, 0])

I = block_interactions(scores_y)
G = scores_y - scores_c
J = block_interactions(G)
def t_interval(values, confidence):
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(len(values))
    critical = stats.t.ppf((1 + confidence) / 2, df=len(values) - 1)
    return mean, (mean - critical * se, mean + critical * se)

j_mean, j_interval_95 = t_interval(J, 0.95)
_, j_interval_90 = t_interval(J, 0.90)
j_resolved = ((j_interval_95[0] > 0 or j_interval_95[1] < 0) and abs(j_mean) >= 0.0625)
j_equivalent = j_interval_90[0] > -0.0625 and j_interval_90[1] < 0.0625
assert I.shape == J.shape == (8,)
assert np.allclose(J, I - block_interactions(scores_c))
assert not (j_resolved and j_equivalent)
print(f"mean I={I.mean():.3f}; mean J={J.mean():.3f}; resolved={j_resolved}; equivalent={j_equivalent}")

## 4. Participant-only and crossed sensitivity bootstraps

Participant-only resampling treats the 32 trained models as fixed and carries each selected participant's full `(8, 2, 2)` slab. The crossed sensitivity also draws complete blocks, never individual model cells. Separate indexing operations preserve the Cartesian crossing of blocks and participants. Both procedures are sensitivities, not replacements for the primary Student $t$ interval over eight observed interactions.

In [ ]:
def bootstrap_interaction(values, replicates, rng, *, resample_blocks):
    values = np.asarray(values, dtype=np.float64)
    B, _, _, P = values.shape
    estimates = np.empty(replicates)
    for b in range(replicates):
        participant_draw = rng.integers(0, P, size=P, endpoint=False)
        sampled = values[..., participant_draw]
        if resample_blocks:
            block_draw = rng.integers(0, B, size=B, endpoint=False)
            sampled = sampled[block_draw]
        assert sampled.shape == values.shape
        estimates[b] = block_interactions(sampled).mean()
    return estimates

bootstrap_rng = np.random.Generator(np.random.PCG64(1501))
participant_only = bootstrap_interaction(scores_y, 2000, bootstrap_rng, resample_blocks=False)
crossed = bootstrap_interaction(scores_y, 2000, bootstrap_rng, resample_blocks=True)
participant_ci = np.quantile(participant_only, [0.025, 0.975], method="linear")
crossed_ci = np.quantile(crossed, [0.025, 0.975], method="linear")
assert participant_ci[0] < participant_ci[1] and crossed_ci[0] < crossed_ci[1]
print("participant-only interval:", participant_ci)
print("crossed interval:         ", crossed_ci)

## 5. The blocks share a finite corpus

The eight pool orderings overlap because they come from one finite GaitLU corpus. Block resampling therefore describes reproducibility over the observed pool ordering, anchor, and optimization construction, conditional on that corpus. It does not simulate eight independently sampled source datasets. Bootstrap draws also do not create new trained models or participants.

## 6. Prospective simulation keeps $n=8$

Before outcome access, simulate exactly eight interaction values under plausible design assumptions. Apply the planned seven-degree-of-freedom interval and the exact materiality rule. Health&Gait outcome aggregates cannot tune the assumed mean, standard deviation, 0.0625 margin, or decision threshold.

In [ ]:
simulation_rng = np.random.Generator(np.random.PCG64(1502))
n_studies, n_model_blocks = 5000, 8
assumed_mean, assumed_sd, margin = -0.08, 0.06, 0.0625
simulated = simulation_rng.normal(assumed_mean, assumed_sd, size=(n_studies, n_model_blocks))
means = simulated.mean(axis=1)
ses = simulated.std(axis=1, ddof=1) / np.sqrt(n_model_blocks)
critical = stats.t.ppf(0.975, df=n_model_blocks - 1)
lower, upper = means - critical * ses, means + critical * ses
resolved_negative = upper < 0
materially_negative = resolved_negative & (means <= -margin)
assert simulated.shape == (5000, 8)
print(f"median 95% half-width={np.median(critical * ses):.3f}")
print(f"P(resolve below zero)={resolved_negative.mean():.3f}")
print(f"P(materially negative)={materially_negative.mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(means, bins=35, color="#6d9f71", edgecolor="white")
ax.axvline(0, color="black", label="zero")
ax.axvline(-margin, color="#9f3f3f", label="materiality margin")
ax.set(xlabel="simulated mean interaction", ylabel="study count",
       title="Prospective eight-block simulation")
ax.legend()
plt.tight_layout()
plt.show()

## Exercises, limits, and takeaways

1. Resample the four cells independently. Explain why the resulting blocks are artificial.
2. Resample participant indices separately by cell. Which repeated-person covariance disappears?
3. Increase the number of bootstrap draws. Why does this not increase the eight model blocks?
4. Change the prospective assumed standard deviation. How do resolution and materiality probabilities respond?

**Takeaway:** exposure, sequence support, temporal support, participant count, and trained-model replication answer different questions. Valid sensitivities carry complete blocks and complete participant profiles. The completion gap preserves the same pairing, and prospective planning never changes the primary model-level $n=8$.

## Continue learning

[Previous notebook: 14](14_paired_inference.ipynb) | [Lecture](../lectures/15_exposure_and_replication.md) | [Curriculum](../README.md) | [Next notebook: 16](16_reproducible_scientific_evaluators.ipynb)